Create RAG by using :
- Ollama app (using llama3.2 model)
- PostgreSQL (database)
- pgvector (extension of postgreSQL for vector type in postgres by install from docker hub of postgres)
- DBeaver (database management)

### Using Ollama app for using LLM model (llama3.2)

In [2]:
import ollama

In [3]:
prompt = "what is the capital of thailand"

response = ollama.chat(
    model="llama3.2",
    messages=[
        # {"role":"system", "content":"You are my assistent"},
        {"role": "user", "content": prompt}
    ]
)

In [4]:
response

ChatResponse(model='llama3.2', created_at='2025-10-02T03:42:48.4621635Z', done=True, done_reason='stop', total_duration=5039889400, load_duration=4428485000, prompt_eval_count=32, prompt_eval_duration=431033200, eval_count=8, eval_duration=176724300, message=Message(role='assistant', content='The capital of Thailand is Bangkok.', thinking=None, images=None, tool_name=None, tool_calls=None))

In [5]:
result = response["message"]["content"]
result

'The capital of Thailand is Bangkok.'

### Connect the database (postgresql)

In [6]:
import psycopg2 # ใช้เชื่อมต่อ postgresql database

In [17]:
conn = psycopg2.connect( # ต้อง run containner ใน docker ไว้
    dbname="mydb",
    user="admin",
    password="1234",
    host="localhost",
    port="5432"
)
cur = conn.cursor() #ใช้ cur ในการ execute ตัว command ต่างๆ ที่มาจาก sql (ใช้เขียนคำสั่ง sql)

In [50]:
# cur.execute("CREATE EXTENSION vector") #CREATE EXTENSION vector เป็นคำสั่งที่ใช้กำหนด vector ให้เป็น type หนึ่งของ table
# conn.commit() # commit เข้า database

เช็คว่า vector enable แล้วรึยัง (เช็คใน DBeaver)
`select * from pg_catalog.pg_extension
where extname = 'vector';`

In [ ]:
# # สร้าง table ชื่อ documents (<ชื่อ column> <type>)
# cur.execute("""
#     CREATE TABLE IF NOT EXISTS documents(
#         id SERIAL PRIMARY KEY,
#         content TEXT,
#         embedding vector(1024)
#     )
# """)
# conn.commit()

In [ ]:
# cur.close()
# conn.close()

### Embedding (text-to-vector)

In [51]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-m3") # model ที่ใช้ embedding

# test
embedding = embedder.encode("สวัสดีค่ะ").tolist()
embedding

[-0.0505496971309185,
 0.027271777391433716,
 -0.054901767522096634,
 -0.019734643399715424,
 -0.008556877262890339,
 -0.05407748371362686,
 -0.003942254930734634,
 -0.04860492795705795,
 0.03668391704559326,
 -0.009794535115361214,
 0.017337718978524208,
 0.014618121087551117,
 0.004314512014389038,
 -0.018172970041632652,
 0.014100722037255764,
 -0.017530763521790504,
 0.01728440821170807,
 -0.03599424660205841,
 -0.007817069068551064,
 -0.04388541728258133,
 -0.025488080456852913,
 0.0025875417049974203,
 0.02201724797487259,
 -0.016907280310988426,
 -0.006039287894964218,
 0.031011803075671196,
 -0.027339471504092216,
 0.0062783160246908665,
 0.018409349024295807,
 -0.02353207767009735,
 0.035705748945474625,
 0.042025912553071976,
 -0.01960710994899273,
 -0.03557587414979935,
 -0.011197238229215145,
 -0.08609771728515625,
 0.00871235877275467,
 -0.018911225721240044,
 -0.0647430568933487,
 0.036221496760845184,
 0.02807416021823883,
 0.002657272620126605,
 0.028483325615525246,
 -

In [12]:
len(embedding)

1024

In [13]:
def add_embedding(text):
    # embedding
    embedding = embedder.encode(text).tolist()
    
    # add embedding to database
    cur.execute("""
        INSERT INTO documents (content, embedding) VALUES (%s, %s)""",
        (text, embedding)
    )
    conn.commit()

In [29]:
documents = [
    "หมามี 4 ขา",
    "คนมี 2 ขา",
    "แมวมี 4 ขา"
]

for doc in documents:
    add_embedding(doc)
    
cur.close()
conn.close()

### Query

In [ ]:
def query_postgreql(query_text, k=5): # k คือ จน.ที่ให้ database return กลับมา
    query_embedding = embedder.encode(query_text).tolist()
    query_embedding_str = "[" + ",".join(map(str, query_embedding)) + "]" # แปลง list ให้เป็น str ให้เหมือนกับใน db
    # print(query_embedding)
    # print(query_embedding_str)
    
    sql_query = """
        SELECT content, embedding <=> %s::vector AS similarity_score
        FROM documents
        ORDER BY similarity_score ASC
        LIMIT %s;
    """
    '''
    Explain :
        - embedding <=> %s::vector คือ เป็นการคำนวณหาค่า similarity ของ vector
        - similarity_score คือตัวแปรที่สร้างไว้เก็บ embedding <=> %s::vector
        - ถ้า similarity_score น้อย แปลว่า ใกล้เคียงมาก
        - ASC เรียงจากน้อยไปมาก (sorting similarity score)
        - %s ตัวแรก แทน query_embedding_str (ต้องตรงกับใน execute)
        - %s ตัวที่สอง แทน k (ต้องตรงกับใน execute)
    '''
    cur.execute(sql_query, (query_embedding_str, k))
    result = cur.fetchall()
    return result

In [28]:
result = query_postgreql("ขาแมวกับขาคนรวมกันได้เท่าไหร่",3)

In [29]:
result

[('แมวมี 4 ขา', 0.26254685297887514),
 ('คนมี 2 ขา', 0.3393824100494385),
 ('หมามี 4 ขา', 0.35193026065826416)]

### Generate response

In [48]:
import ollama

def generate_response(query_text):
    
    # ค้นข้อมูลที่ใกล้เคียงกับคำถาม 3 อันดับแรก
    retrieved_docs = query_postgreql(query_text, 3)
    
    # แสดงแค่ข้อมูล content
    context = ",".join(doc[0] for doc in retrieved_docs)
    # print(context)
    
    prompt = f"""Answer the question based on the following 
    context: {context}
    Question: {query_text}
    """
    print(prompt)
    
    response = ollama.chat(
        model="llama3.2",
        messages=[
            {"role":"system", "content":"You are an assistent."},
            {"role": "user", "content": prompt}
        ]
    )
    
    return response["message"]["content"]    

In [49]:
generate_response("ขาแมวกับขาคนรวมกันได้เท่าไหร่")


Answer the question based on the following 
    context: แมวมี 4 ขา,คนมี 2 ขา,หมามี 4 ขา
    Question: ขาแมวกับขาคนรวมกันได้เท่าไหร่
    


'ตามที่กล่าวไว้แล้ว\n\n* แมว มีขา 4 ขา\n* คน มีขา 2 ขา\n\nดังนั้น หากเราบวกขาของแมวและคนเข้าด้วยกัน จะได้รวมขา 6 ขา (4 ขาของแมว + 2 ขาของคน)'

In [52]:
cur.close()
conn.close()

In [41]:
select_status = "3"
status = {"1":"Add Embedding", "2":"RAG"}

In [25]:
type(status)

dict

In [39]:
for key in status.keys():
    print(key)

1
2


In [ ]:
if select_status in status.keys() and :

SyntaxError: expected ':' (849187476.py, line 1)